# Proyecto Final — Conditional Imitation Learning (CIL)
## MR4010.10 Navegacion Autonoma — Maestria en IA, Tec de Monterrey

Entrenamiento del modelo CIL ramificado (Codevilla et al., arXiv:1710.02410) con backbone tipo Bojarski (arXiv:1604.07316).

El dataset de conduccion manual (imagenes + `driving_log.csv`) se aloja en un repositorio de GitHub y se clona en la primera celda. El modelo predice el angulo de direccion condicionado al comando de navegacion (FOLLOW / LEFT / STRAIGHT / RIGHT). La velocidad NO forma parte del entrenamiento.

**Flujo:** clonar dataset y codigo -> cargar -> data augmentation (flip + brillo) -> entrenar -> evaluar (MAE por comando) -> exportar `.keras` y `.tflite`.

## 1. Entorno y GPU

In [ ]:
import tensorflow as tf
print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Clonar el dataset y el codigo de entrenamiento

El dataset crudo (imagenes + `driving_log.csv`) vive en un repositorio publico de GitHub. Ajusta la URL si cambia el nombre de tu repo.

In [ ]:
# Dataset crudo de conduccion manual (debe contener driving_log.csv e IMG/)
!git clone https://github.com/oscar-ramirez-anaya/cil_dataset

# Codigo de entrenamiento (provee train_cil.py para reutilizar sus funciones)
!git clone https://github.com/oscar-ramirez-anaya/navegacion_autonoma_final

import sys
sys.path.append('/content/navegacion_autonoma_final/cil_training')

DATA_DIR = '/content/cil_dataset'
!ls -la $DATA_DIR

## 3. Cargar el dataset

Se reutilizan las funciones de `train_cil.py` para garantizar que el preprocesamiento (recorte, resize 66x200, normalizacion) sea identico al usado por el controlador del Mundo #2 durante la inferencia.

In [ ]:
import train_cil as cil
import numpy as np
import matplotlib.pyplot as plt

X, C, y = cil.load_dataset(DATA_DIR)
print('Imagenes:', X.shape, '| Comandos:', C.shape, '| Angulos:', y.shape)

## 4. Data augmentation (flip + brillo)

El flip horizontal niega el angulo e intercambia el comando LEFT<->RIGHT. Sube `brightness_copies` para superar las 10 mil imagenes si tu dataset crudo es pequeno.

In [ ]:
Xa, Ca, ya = cil.augment(X, C, y, brightness_copies=1)
print('Total tras augmentation:', len(Xa))
assert len(Xa) > 10000, 'El dataset final debe superar las 10 mil imagenes (sube brightness_copies o recolecta mas).'

# Histograma de angulos para inspeccionar el balance izquierda/derecha
plt.figure(figsize=(6,3))
plt.hist(ya, bins=41)
plt.title('Distribucion del angulo de direccion (aumentado)')
plt.xlabel('steering (rad)'); plt.ylabel('frecuencia'); plt.show()

## 5. Construir el modelo CIL ramificado

In [ ]:
model = cil.construir_cil()
model.summary()

## 6. Entrenamiento

In [ ]:
history, val_data = cil.entrenar(model, Xa, Ca, ya, epochs=40, batch_size=64)

## 7. Curvas de entrenamiento y evaluacion por comando

In [ ]:
h = history.history
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].plot(h['loss'], label='train'); ax[0].plot(h['val_loss'], label='val')
ax[0].set_title('Perdida (MSE)'); ax[0].set_xlabel('epoca'); ax[0].legend()
ax[1].plot(h['mae'], label='train'); ax[1].plot(h['val_mae'], label='val')
ax[1].set_title('MAE de steering'); ax[1].set_xlabel('epoca'); ax[1].legend()
plt.show()

cil.evaluar(model, val_data)

## 8. Exportacion (.keras + .tflite) y verificacion de paridad

In [ ]:
keras_path, tflite_path = cil.exportar(model, val_data)

# Descargar el modelo TFLite para copiarlo al controlador del Mundo #2
from google.colab import files
files.download(tflite_path)

## 9. Integracion en Webots (Mundo #2)

1. Copia `cil_model.tflite` a `navegacion_autonoma_final/controllers/cil_autonomous/model/`.
2. Abre `worlds/proyecto_final_mundo2.wbt` en Webots y asigna el controlador `cil_autonomous` al BmwX5.
3. Durante la simulacion, el operador proporciona el comando de navegacion por teclado (`1`=FOLLOW, `2`=LEFT, `3`=STRAIGHT, `4`=RIGHT); el modelo predice el angulo de direccion y las capas de seguridad (evasion, peaton, radar) tienen prioridad sobre el modelo.